# Notebook 06: Error Analysis, Explainability & Fairness Auditing
Qualitative analysis of the top 20 False Positives and False Negatives, feature importance with Permutation Importance & SHAP, and demographic fairness slices.

In [1]:
import os, sys
from pathlib import Path
root = Path.cwd() if (Path.cwd() / 'configs').exists() else Path.cwd().parent
os.chdir(root)
if str(root / 'src') not in sys.path:
    sys.path.insert(0, str(root / 'src'))
import pandas as pd
import joblib
from bechdel.eval.fairness import audit_model_fairness
from bechdel.features.builder import get_metadata_feature_names, get_dialogue_feature_names

tier2_df = pd.read_parquet('data/processed/tier2_features.parquet')
errors_df = pd.read_csv('reports/error_analysis_top20.csv')
print(f'Total discrepancy cases: {len(errors_df)}')
print('Sample False Positives (Detector=Pass, Crowd=Fail):')
display(errors_df[errors_df['error_type'].str.contains('Positive')][['title_cornell', 'bechdel_rating', 'primary_reason']].head(5))
print('Sample False Negatives (Detector=Fail, Crowd=Pass):')
display(errors_df[errors_df['error_type'].str.contains('Negative')][['title_cornell', 'bechdel_rating', 'primary_reason']].head(5))

Total discrepancy cases: 40
Sample False Positives (Detector=Pass, Crowd=Fail):


,title_cornell,bechdel_rating,primary_reason
0,an american werewolf in london,2,Crowd labeled 2 (talked only about a man): pro...
1,austin powers: international man of mystery,2,Crowd labeled 2 (talked only about a man): pro...
2,big fish,1,Crowd labeled 1 (women do not talk): detector ...
3,the bourne supremacy,2,Crowd labeled 2 (talked only about a man): pro...
4,braveheart,2,Crowd labeled 2 (talked only about a man): pro...


Sample False Negatives (Detector=Fail, Crowd=Pass):


,title_cornell,bechdel_rating,primary_reason
20,amadeus,3,Stage B failed: Cornell dialogue transcript om...
21,basic,3,Stage A failed: only 1 confirmed female chars ...
22,blast from the past,3,Stage B failed: Cornell dialogue transcript om...
23,highlander,3,Stage A failed: only 1 confirmed female chars ...
24,human nature,3,Stage B failed: Cornell dialogue transcript om...


In [2]:
# Fairness audit across Decades and Genres
best_pipe = joblib.load('artifacts/models/best_classifier.joblib')
features = [c for c in (get_metadata_feature_names() + get_dialogue_feature_names()) if c in tier2_df.columns]
tier2_df['pred'] = best_pipe.predict(tier2_df[features])
fairness_audit = audit_model_fairness(tier2_df)
print('Fairness Audit by Decade:')
display(fairness_audit.get('decade'))

Fairness Audit by Decade:


,group_feature,group_value,support,ground_truth_pass_rate,accuracy,precision,recall,f1,f1_drop_from_overall,recall_drop_from_overall,significant_drop
0,decade,1930,10,0.6000,1.0000,1.0000,1.0000,1.0000,-0.2911,-0.3422,False
1,decade,1950,15,0.4667,0.7333,1.0000,0.4286,0.6000,0.1089,0.2292,True
2,decade,1960,14,0.2143,0.9286,0.7500,1.0000,0.8571,-0.1482,-0.3422,False
3,decade,1970,35,0.3143,0.8286,0.7778,0.6364,0.7000,0.0089,0.0214,False
4,decade,1980,79,0.5190,0.7089,0.7812,0.6098,0.6849,0.0240,0.0480,False
5,decade,1990,155,0.4903,0.7419,0.7571,0.6974,0.7260,-0.0171,-0.0396,False
6,decade,2000,85,0.4353,0.6941,0.6897,0.5405,0.6061,0.1029,0.1172,False
